In [1]:
import os
import sys
import pandas as pd

import logging

logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)


%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


'c:\\Users\\fiscarelli\\Desktop\\Progetti\\Manpower IT\\Auto-Match\\Candidates-to-Jobs-Auto-Match-Cortex-AI'

In [2]:
from src.autoMatch.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DeploymentConfig:
    root_dir: str
    database: str
    schema: str
    warehouse: str
    config_dir: str
    stage_sp: str
    data_ingestion_sp: str
    data_transformation_sp: str
    data_ingestion_task: str
    data_transformation_task: str
    candidate_log_table: str
    vacancy_log_table: str
    runtime_version: str
    data_ingestion_imports: list[str]
    data_ingestion_packages: list[str]
    data_transformation_imports: list[str]
    data_transformation_packages: list[str]


In [4]:
from src.autoMatch.constants import *
from src.autoMatch.utils.common import read_yaml, create_directories
from src.autoMatch import logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):


        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        #self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_deployment_config(self) -> DeploymentConfig:
        config = self.config.deployment
        params = self.params.deployment

        #create_directories([config.root_dir])

        deployment_config = DeploymentConfig(
            root_dir=config.root_dir,
            database=config.database,
            schema=config.schema,
            warehouse=config.warehouse,
            config_dir=config.config_dir,
            stage_sp=config.stage_sp,
            data_ingestion_sp=config.data_ingestion_sp,
            data_transformation_sp=config.data_transformation_sp,
            data_ingestion_task=config.data_ingestion_task,
            data_transformation_task=config.data_transformation_task,
            candidate_log_table=self.config.candidate_log_table,
            vacancy_log_table=self.config.vacancy_log_table,
            runtime_version=params.runtime_version,
            data_ingestion_imports=params.data_ingestion_imports,
            data_ingestion_packages=params.data_ingestion_packages,
            data_transformation_imports=params.data_transformation_imports,
            data_transformation_packages=params.data_transformation_packages
        )

        return deployment_config

In [5]:
from src.autoMatch.utils.snowflake_utils import generate_create_stage_command

from src.autoMatch.pipeline.stage_01_data_ingestion import run_data_ingestion_pipeline
from src.autoMatch.pipeline.stage_03_data_transformation import run_data_transformation_pipeline

class Deployment:
    def __init__(self, config: DeploymentConfig):
        self.config = config

    def create_stage(self, session, is_permanent=True, overwrite=False):
        
        stage_sp = self.config.stage_sp

        if overwrite:
            session.sql(f"DROP STAGE IF EXISTS {stage_sp}").collect()
        session.sql(
            generate_create_stage_command(
                stage_sp,
                is_permanent=is_permanent,
                overwrite=True
            )
        ).collect()
        logger.info(f"Stage {stage_sp} successfully created.")


    def upload_config_files(self, session):
        """
        Upload all YAML files from a local config directory to a Snowflake stage.
        """

        config_dir = self.config.config_dir
        stage_sp = self.config.stage_sp
        local_config_dir = Path(config_dir)

        for file in local_config_dir.glob("*.yaml"):
            session.file.put(
                str(file),
                stage_sp,
                auto_compress=False,
                overwrite=True
            )
            logger.info(f"File {str(file)} uploaded on stage {stage_sp}.")



    def deploy_data_ingestion_stored_procedure(self, session):

        stage_sp = self.config.stage_sp
        data_ingestion_sp = self.config.data_ingestion_sp
        runtime_version=self.config.runtime_version
        data_ingestion_imports=self.config.data_ingestion_imports
        data_ingestion_packages=self.config.data_ingestion_packages

        session.sql(f"DROP PROCEDURE IF EXISTS {data_ingestion_sp}();").collect()
        session.sproc.register(
            run_data_ingestion_pipeline, 
            name = data_ingestion_sp,
            is_permanent = True,
            stage_location = stage_sp,
            replace = True,
            runtime_version = runtime_version,
            imports =  data_ingestion_imports,
            packages = data_ingestion_packages
        )
        logger.info(f"Stored procedure {data_ingestion_sp} successfully deployed on stage {stage_sp}.")

        

    def deploy_data_transformation_stored_procedure(self, session):

        stage_sp = self.config.stage_sp
        data_transformation_sp = self.config.data_transformation_sp
        runtime_version=self.config.runtime_version
        data_transformation_imports=self.config.data_transformation_imports
        data_transformation_packages=self.config.data_transformation_packages
        
        session.sql(f"DROP PROCEDURE IF EXISTS {data_transformation_sp}();").collect()
        session.sproc.register(
            run_data_transformation_pipeline, 
            name = data_transformation_sp,
            is_permanent = True,
            stage_location = stage_sp,
            replace = True,
            runtime_version = runtime_version,
            imports =  data_transformation_imports,
            packages = data_transformation_packages
        )
        logger.info(f"Stored procedure {data_transformation_sp} successfully deployed on stage {stage_sp}.")


    def create_tasks(self, session):
        database = self.config.database
        schema = self.config.schema
        warehouse = self.config.warehouse
        data_ingestion_sp = self.config.data_ingestion_sp
        data_transformation_sp = self.config.data_transformation_sp
        data_ingestion_task = self.config.data_ingestion_task
        data_transformation_task = self.config.data_transformation_task

        tasks = [data_ingestion_task, data_transformation_task]
        # Suspend existing tasks if they exist
        for task_name in tasks:
            result = session.sql(f"SHOW TASKS LIKE '{task_name}'").collect()
            session.sql(f"ALTER TASK {task_name} SUSPEND").collect() if result else print(f"Task {task_name} does not exist.")

        data_ingestion_task_sql = f"""
            CREATE OR REPLACE TASK {data_ingestion_task}
            WAREHOUSE = '{warehouse}'
            SCHEDULE = 'USING CRON 0/2 9-19 * * 1-5 CET'
            --SCHEDULE = 'USING CRON 0 8 * * 1-5 CET'
            AS
            CALL {database}.{schema}.{data_ingestion_sp}();
        """
        session.sql(data_ingestion_task_sql).collect()
        logger.info(f"Task {data_ingestion_task} successfully created.")
        
        data_transformation_task_sql = f"""
            CREATE OR REPLACE TASK {data_transformation_task}
            WAREHOUSE = '{warehouse}'
            AFTER {data_ingestion_task}
            AS
            CALL {database}.{schema}.{data_transformation_sp}();
        """
        session.sql(data_transformation_task_sql).collect()
        logger.info(f"Task {data_transformation_task} successfully created.")

        # Resume all tasks
        for task_name in reversed(tasks):
            session.sql(f"ALTER TASK {task_name} RESUME").collect()
            logger.info(f"Task {task_name} resumed")


    def drop_tasks(self, session):
        data_ingestion_task = self.config.data_ingestion_task
        data_transformation_task = self.config.data_transformation_task

        tasks = [data_ingestion_task, data_transformation_task]

        # Suspend existing tasks if they exist
        logger.info(f"Deleting tasks...")
        for task_name in tasks:
            result = session.sql(f"SHOW TASKS LIKE '{task_name}'").collect()
            if result:
                session.sql(f"ALTER TASK {task_name} SUSPEND").collect()
                session.sql(f"DROP TASK {task_name}").collect()
                logger.info(f"Task {task_name} successfully deleted.")
            else:
                logger.info(f"Task {task_name} does not exist yet.")

    def reset_pipeline(self, session):
        """
        Resets inference pipeline
        """
        candidate_log_table = self.config.candidate_log_table
        vacancy_log_table = self.config.vacancy_log_table

        session.sql(f"DROP TABLE IF EXISTS {candidate_log_table}").collect()
        session.sql(f"DROP TABLE IF EXISTS {vacancy_log_table}").collect()



In [6]:
try:
    config = ConfigurationManager()
    deployment_config = config.get_deployment_config()
    deployment = Deployment(config=deployment_config)
    deployment.create_stage(session)
    deployment.upload_config_files(session)
    deployment.deploy_data_ingestion_stored_procedure(session)
    deployment.deploy_data_transformation_stored_procedure(session)
    deployment.reset_pipeline(session)
    deployment.drop_tasks(session)
    deployment.create_tasks(session)
except Exception as e:
    raise e

[2026-03-17 11:20:29,791: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\Auto-Match\Candidates-to-Jobs-Auto-Match-Cortex-AI\config\config_test.yaml loaded successfully]
[2026-03-17 11:20:29,795: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\Auto-Match\Candidates-to-Jobs-Auto-Match-Cortex-AI\config\params.yaml loaded successfully]
[2026-03-17 11:20:29,797: INFO: common: created directory at: artifacts_b]
[2026-03-17 11:20:30,028: INFO: 1870244256: Stage MPG_IT_AUTOMATCH_STAGE_SP successfully created.]
[2026-03-17 11:20:31,116: INFO: 1870244256: File config\config_dev.yaml uploaded on stage MPG_IT_AUTOMATCH_STAGE_SP.]
[2026-03-17 11:20:31,538: INFO: 1870244256: File config\config_test.yaml uploaded on stage MPG_IT_AUTOMATCH_STAGE_SP.]
[2026-03-17 11:20:32,037: INFO: 1870244256: File config\params.yaml uploaded on stage MPG_IT_AUTOMATCH_STAGE_SP.]
[2026-03-17 11:20:32,419: INFO: 1870244256: File config\schema.yaml uploaded on stage MP

In [7]:
#session.call("DataIngestionSP")
#session.call("DataTransformationSP")

In [8]:
session.sql("LIST @MPG_IT_AUTOMATCH_STAGE_SP;").collect()

[Row(name='mpg_it_automatch_stage_sp/config_dev.yaml', size=1760, md5='ddc498bdfe3b063dc331634c2756acb2', last_modified='Tue, 17 Mar 2026 10:20:31 GMT'),
 Row(name='mpg_it_automatch_stage_sp/config_test.yaml', size=1984, md5='c36e6bfcacc75bdd859c60ffbbabd8e1', last_modified='Tue, 17 Mar 2026 10:20:31 GMT'),
 Row(name='mpg_it_automatch_stage_sp/f68abd0c067d23557b7352aa30712cfd751881456c4b09811d0659353eb158a2/src.zip', size=51536, md5='d114e12c3cd722c290462cf8cc862922', last_modified='Tue, 17 Mar 2026 10:20:35 GMT'),
 Row(name='mpg_it_automatch_stage_sp/params.yaml', size=464, md5='a178c3c64406bcee7ed52e5494a3e454', last_modified='Tue, 17 Mar 2026 10:20:31 GMT'),
 Row(name='mpg_it_automatch_stage_sp/schema.yaml', size=42960, md5='7f192b82fd76bc05450c19fb01ff5191', last_modified='Tue, 17 Mar 2026 10:20:32 GMT')]